In [ ]:
from joblib import load
from text_preprocessing import _load_data
from codecarbon import EmissionsTracker
import csv
import time
import random


from text_classification import generate_model, model_validation
from modify_dataset import modify_dataset_and_raw_data_with_percentage_size_to_keep
from modify_dataset import modify_dataset_select_features

from sklearn.tree import DecisionTreeClassifier
from energy_utils import energy_stats


In [ ]:
raw_data = _load_data()
preprocessed_data = load('output/preprocessed_data.joblib')

In [ ]:
classifier = DecisionTreeClassifier()

modified_preprocessed_data, modified_raw_data = modify_dataset_and_raw_data_with_percentage_size_to_keep(
        preprocessed_data, raw_data, 100)

In [ ]:
training_tracker = EmissionsTracker(save_to_file=False, allow_multiple_runs=True, log_level="error")
#### START TIMED TRAINING SECTION ####
training_tracker.start()
classifier, X_train, X_test, y_train, y_test, test_messages = generate_model(classifier, modified_raw_data,
                                                                                 modified_preprocessed_data)
training_energy_consumption_kwh = training_tracker.stop()
#### STOP TIMED TRAINING SECTION ####
training_energy_consumption, training_duration = energy_stats(training_energy_consumption_kwh, training_tracker)
_, scores, report = model_validation(classifier, X_test, y_test)

print(f"  Energy Consumption: {training_energy_consumption:.4f} Joules")
print(f"  Duration: {training_duration:.2f} seconds")
print(f"  F1 Score: {scores['f1']:.2f}")
print(f"  Accuracy: {scores['accuracy']:.2f}")